# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs only a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week1day2free.ipynb">week1day2free.ipynb</a>.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it! There's some interesting content on Jev for the pros.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [1]:
import sys
print(sys.executable)

E:\python\learning\Udemy_FDE\.venv\Scripts\python.exe


In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [8]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("GPT_API_KEY")
OPENROUTER_BASE_URL = "https://api.openai.com/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

response = openrouter.chat.completions.create(model="gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

Arrr, there’s no single agreed-upon meaning of the universe. Science tells us how it works; philosophy and faith offer different ideas about why it exists. And many reckon meaning isn’t hidden out there like buried treasure—it’s something we make through love, curiosity, kindness, and the lives we share.

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [10]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [11]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [12]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [13]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

### Explore the data

It's always good to dig into the data before running experiments.

In [14]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


Claim CLM-1015 might be a familiar example...

In [16]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [17]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [18]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [19]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [20]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [21]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [26]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [29]:
first = decide(claims[0], "gpt-6-luna", "low")
first

{'model': 'gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'Approve under section 3: the carrier canceled the flight for a crew problem, and the delay from the scheduled departure on June 18 until the actual departure on June 19 exceeded the Premium plan’s 6-hour waiting period. The required carrier confirmation and receipts are attached; the $420 in reasonable accommodation, meal, and local transport costs is below the $800 maximum, and no deductible applies.',
 'seconds': 3.3}

### Now running multiple models

In [30]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    #cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [31]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

BadRequestError: Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [33]:
luna_medium = run("gpt-6-luna", "medium", claims, labels)

gpt-6-luna at medium: 18 of 25 right in 108 seconds


In [ ]:
# Add your own experiments here!

In [ ]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [32]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

NameError: name 'luna_low' is not defined

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [34]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

NameError: name 'luna_low' is not defined

### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [ ]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Optional extras

These two experiments on the queue are optional. Nothing later in the lab depends on them.

### Sol at high

GPT-6 Sol is Luna's bigger sibling; we run it at the `high` reasoning level to see how much better it does on this prompt.

In [ ]:
sol = run("openai/gpt-6-sol", "high", claims, labels)

### Jev

Jev, from TypeSafe AI, doesn't generate text. We send it the facts as `state` along with a typed question, and it returns a probability for each option and a confidence. It chooses between approve, decline and refer in a fraction of a second. It can't write a rationale or work out a payout, so we score it on the decision alone.

Jev is on OpenRouter's Decisions endpoint, which the OpenAI client can't reach. We use OpenRouter's own Python client with the same key.

In [ ]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[claim["claim_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [ ]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result["decision"] == labels[result["claim_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Here are Jev's decisions split at a confidence of 0.9. One way to use a fast model like this is to keep the answers it's confident about and send the rest to a stronger model.

In [ ]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [35]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses Luna at `low` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with Luna at `low`.

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [ ]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, sol, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You compared several models and an agent on UnderCover's claims, with accuracy, cost and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>